# TrekTrak — Pedestrian vs Cyclist Classifier

This notebook demonstrates how a Random Forest classifier could be trained to
distinguish pedestrians from cyclists using per-detection-event sensor features.

**On-device vision:** the trained model (or a compressed version) would run on the
ESP32 sensor node to label each detection event at the edge before publishing to MQTT,
removing the need for post-hoc classification in the pipeline.

**Honest caveat:** All training data below is synthetically generated from assumed
physical distributions. A real deployment would require labeled field recordings
where an observer tags each detection as pedestrian or cyclist.

## Features (per detection event)
| Feature | Description | Pedestrian | Cyclist |
|---|---|---|---|
| `dwell_sec` | Seconds the object stayed in range | Long (~4.5 s) | Short (~1.2 s) |
| `peak_proximity_cm` | Closest measured distance (cm) | Farther (~30 cm) | Closer (~18 cm) |
| `pir_triggered` | Did the PIR also fire? (0/1) | Reliably (85%) | Less reliable (60%) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score
)
from sklearn.dummy import DummyClassifier
import joblib

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')
SEED = 42
np.random.seed(SEED)

## 1  Synthetic Training Data

Each row represents one detection event logged by the ESP32.
In a real deployment this table would come from annotated field sessions.

In [ ]:
N = 500  # events per class

def make_pedestrian(n):
    return pd.DataFrame({
        'dwell_sec':          np.clip(np.random.normal(4.5, 1.5, n), 1.0, 12.0),
        'peak_proximity_cm':  np.clip(np.random.normal(30,  12,  n), 10,  90.0),
        'pir_triggered':      np.random.binomial(1, 0.85, n).astype(float),
        'label': 0,  # pedestrian
    })

def make_cyclist(n):
    return pd.DataFrame({
        'dwell_sec':          np.clip(np.random.normal(1.2, 0.4,  n), 0.3,  3.0),
        'peak_proximity_cm':  np.clip(np.random.normal(18,  8.0,  n), 5.0, 60.0),
        'pir_triggered':      np.random.binomial(1, 0.60, n).astype(float),
        'label': 1,  # cyclist
    })

df = pd.concat([make_pedestrian(N), make_cyclist(N)], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f'Dataset: {len(df)} events  ({df.label.value_counts()[0]} pedestrian, {df.label.value_counts()[1]} cyclist)')
df.describe().round(2)

## 2  Feature Distributions by Class

In [ ]:
CLASS_NAMES = {0: 'Pedestrian', 1: 'Cyclist'}
COLORS      = {0: 'steelblue', 1: 'darkorange'}
FEATURES    = ['dwell_sec', 'peak_proximity_cm', 'pir_triggered']

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, feat in zip(axes, FEATURES):
    for cls in [0, 1]:
        ax.hist(df[df.label == cls][feat], bins=20, alpha=0.6,
                label=CLASS_NAMES[cls], color=COLORS[cls], edgecolor='white')
    ax.set_title(feat)
    ax.set_xlabel('Value')
    ax.legend()
plt.suptitle('Feature Distributions by Class', y=1.02)
plt.tight_layout()
plt.savefig('figures/ml_feature_distributions.png', bbox_inches='tight')
plt.show()

## 3  Train / Test Split and Random Forest

In [ ]:
X = df[FEATURES]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
print(f'Train: {len(X_train)}  Test: {len(X_test)}')

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=SEED)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f'\nAccuracy : {accuracy_score(y_test, y_pred):.3f}')
print(f'F1 macro : {f1_score(y_test, y_pred, average="macro"):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=['Pedestrian', 'Cyclist']))

## 4  Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['Pedestrian', 'Cyclist']
).plot(ax=ax, colorbar=False)
ax.set_title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.savefig('figures/confusion_matrix.png', bbox_inches='tight')
plt.show()

## 5  Feature Importances

In [ ]:
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(6, 3))
imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest — Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('figures/feature_importances.png', bbox_inches='tight')
plt.show()
print(imp.round(3).to_string())

## 6  Comparison: SVM Alternative

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

svm = SVC(kernel='rbf', C=1.0, random_state=SEED)
svm.fit(X_train_s, y_train)
y_svm = svm.predict(X_test_s)

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
y_base = baseline.predict(X_test)

results = pd.DataFrame({
    'Model':    ['Majority-class baseline', 'SVM (RBF)', 'Random Forest'],
    'Accuracy': [accuracy_score(y_test, y_base),
                 accuracy_score(y_test, y_svm),
                 accuracy_score(y_test, y_pred)],
    'F1 macro': [f1_score(y_test, y_base, average='macro', zero_division=0),
                 f1_score(y_test, y_svm,  average='macro'),
                 f1_score(y_test, y_pred, average='macro')],
}).set_index('Model')
results.round(3)

## 7  On-Device Deployment Notes

To run inference on the ESP32 sensor node at the edge:

| Option | Approach | Toolchain |
|---|---|---|
| **emlearn** | Converts sklearn model to C header | `pip install emlearn` → `emlearn.convert(rf)` |
| **TF Lite** | Retrain with Keras, quantise to int8 | `tflite_convert`, TFLite Micro for ESP32 |
| **Threshold rule** | Hard-code dwell threshold ~2.5 s | No ML needed; simple but less accurate |

The ESP32 has 520 KB SRAM. A shallow Random Forest (depth ≤ 8, 100 trees) compiles
to approximately 20–40 KB of C code with emlearn, which fits comfortably.

In [ ]:
# Save model for offline use / pipeline integration
joblib.dump(rf, 'models/rf_pedestrian_cyclist.joblib')
print('Model saved → analysis/models/rf_pedestrian_cyclist.joblib')
print()
print('Example single-event inference:')
example = pd.DataFrame([{'dwell_sec': 0.9, 'peak_proximity_cm': 20, 'pir_triggered': 0}])
pred = rf.predict(example)[0]
prob = rf.predict_proba(example)[0]
print(f'  Input : {example.to_dict(orient="records")[0]}')
print(f'  Result: {CLASS_NAMES[pred]}  (confidence: {max(prob):.0%})')